In [ ]:
from datetime import datetime
import json

with open("/Users/enovikov11/Desktop/data/mac/rtx-data/Wolt_2025-08-13T20_45_44.739Z.json", "r") as file:
    wolt = json.load(file)

items = []

orders = [(datetime.fromtimestamp(o["delivery_time"]["$date"] / 1000).strftime('%Y-%m-%d'),
 o["payment_amount"] / 100) for o in wolt["orders"] if "delivery_time" in o and o["currency"] == "RSD"]

for order in wolt["orders"]:
    if "delivery_time" not in order or order["currency"] != "RSD":
        continue

    for item in order["items"]:
        items.append((order["venue_name"], item["name"], item["count"], item["price"] / 100))

len(orders), sum(s for dt, s in orders)

In [ ]:
days_active = len(set(dt for dt, s in orders))
total_spent = int(sum(s for dt, s in orders))

total_spent, days_active, int(total_spent / days_active)

In [ ]:
import pandas as pd
# Create DataFrame and aggregate as before
df = pd.DataFrame(items, columns=['venue', 'item', 'count', 'price'])
result = df.groupby(['venue', 'item']).agg({
    'count': 'sum',
    'price': lambda x: (x * df.loc[x.index, 'count']).sum() / df.loc[x.index, 'count'].sum()
}).rename(columns={'count': 'sum_count', 'price': 'avg_price'}).reset_index()

# Add total spend column
result['spend_total'] = result['sum_count'] * result['avg_price']

# Sort by spend_total descending first
result = result.sort_values('spend_total', ascending=False)

# Group by item, sum counts/spend, and take first venue (highest spend)
final = result.groupby('item').agg({
    'venue': 'first',
    'sum_count': 'sum',
    'spend_total': 'sum',
    'avg_price': lambda x: (x * result.loc[x.index, 'sum_count']).sum() / result.loc[x.index, 'sum_count'].sum()
}).reset_index()

# Re-sort by total spend descending
final = final.sort_values('spend_total', ascending=False)
# final = final.sort_values('sum_count', ascending=False)

# Display all rows
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 40)

# Reorder columns - use 'final', not 'df'
df_final = final[['venue', 'item', 'sum_count', 'avg_price', 'spend_total']]

# Convert to int and format
df_final['avg_price'] = df_final['avg_price'].round().astype(int)
df_final['spend_total'] = df_final['spend_total'].astype(int)

# Display version with comma formatting
df_final_display = df_final.copy()
df_final_display['spend_total'] = df_final_display['spend_total'].apply(lambda x: f"{x:,}")

hd = df_final_display.head(25)

total_subsum = (hd["sum_count"] * hd["avg_price"]).sum()
total_sum = sum(s for dt, s in orders)

print(total_subsum, 100 * total_subsum / total_sum)

hd

In [ ]:
# monthly
from collections import defaultdict
from datetime import datetime
import matplotlib.pyplot as plt

def analyze_monthly_data(orders):
    # Group by yyyy-mm and sum amounts + count transactions
    monthly_totals = defaultdict(float)
    monthly_counts = defaultdict(int)
    
    for date_str, amount in orders:
        # Extract yyyy-mm from date string
        month_key = date_str[:7]  # '2025-01-15' -> '2025-01'
        monthly_totals[month_key] += amount
        monthly_counts[month_key] += 1
    
    # Find date range
    if not orders:
        return {}, [], {}, [], {}
    
    dates = [datetime.strptime(date_str, '%Y-%m-%d') for date_str, _ in orders]
    min_date = min(dates).replace(day=1)
    max_date = max(dates).replace(day=1)
    
    # Generate all months in range
    current = min_date
    all_months = []
    while current <= max_date:
        month_str = current.strftime('%Y-%m')
        all_months.append(month_str)
        # Move to next month
        if current.month == 12:
            current = current.replace(year=current.year + 1, month=1)
        else:
            current = current.replace(month=current.month + 1)
    
    # Create complete datasets with zeros and calculate averages
    complete_spending = []
    complete_counts = []
    complete_averages = []
    
    for month in all_months:
        amount = monthly_totals.get(month, 0.0)
        count = monthly_counts.get(month, 0)
        avg_price = amount / count if count > 0 else 0.0
        
        complete_spending.append((month, amount))
        complete_counts.append((month, count))
        complete_averages.append((month, avg_price))
    
    # Sort by month (already sorted from generation)
    complete_spending.sort()
    complete_counts.sort()
    complete_averages.sort()
    
    return (dict(complete_spending), complete_spending, 
            dict(complete_counts), complete_counts,
            dict(complete_averages), complete_averages)

def plot_triple_analysis(spending_data, count_data, avg_data):
    months_s, amounts = zip(*spending_data) if spending_data else ([], [])
    months_c, counts = zip(*count_data) if count_data else ([], [])
    months_a, averages = zip(*avg_data) if avg_data else ([], [])
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # Top plot: Spending and Order Count (dual y-axis)
    bars = ax1.bar(months_s, amounts, color='steelblue', alpha=0.7, label='Spending (RSD)')
    ax1.set_ylabel('Amount RSD', color='steelblue')
    ax1.tick_params(axis='y', labelcolor='steelblue')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    ax1.grid(axis='y', alpha=0.3)
    
    # Transaction count line on secondary y-axis
    ax1_twin = ax1.twinx()
    line = ax1_twin.plot(months_c, counts, marker='o', color='darkred', linewidth=2,
                        markersize=6, label='Order Count')
    ax1_twin.set_ylabel('Number of Orders', color='darkred')
    ax1_twin.tick_params(axis='y', labelcolor='darkred')
    ax1_twin.set_ylim(bottom=0)
    
    ax1.set_title('Monthly Wolt Spending and Order Count')
    
    # Combined legend for top plot
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1_twin.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    # Bottom plot: Average Transaction Price
    ax2.plot(months_a, averages, marker='s', color='green', linewidth=2,
             markersize=6, label='Avg Price per Order')
    ax2.set_xlabel('Month')
    ax2.set_ylabel('Average Price (RSD)', color='green')
    ax2.tick_params(axis='y', labelcolor='green')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    ax2.grid(True, alpha=0.3)
    ax2.set_title('Average Transaction Price')
    ax2.legend()
    
    # Format x-axis for both plots
    for ax in [ax1, ax2]:
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

# Alternative: Single plot with three metrics
def plot_single_triple_analysis(spending_data, count_data, avg_data):
    months_s, amounts = zip(*spending_data) if spending_data else ([], [])
    months_c, counts = zip(*count_data) if count_data else ([], [])
    months_a, averages = zip(*avg_data) if avg_data else ([], [])
    
    fig, ax1 = plt.subplots(figsize=(14, 8))
    
    # Spending bars
    bars = ax1.bar(months_s, amounts, color='steelblue', alpha=0.6, 
                   label='Total Spending (RSD)', width=0.6)
    ax1.set_xlabel('Month')
    ax1.set_ylabel('Amount RSD', color='steelblue')
    ax1.tick_params(axis='y', labelcolor='steelblue')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    
    # Order count on second y-axis
    ax2 = ax1.twinx()
    line1 = ax2.plot(months_c, counts, marker='o', color='darkred', linewidth=2,
                     markersize=6, label='Order Count')
    
    # Average price on third y-axis (sharing with order count but different scale)
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    line2 = ax3.plot(months_a, averages, marker='s', color='green', linewidth=2,
                     markersize=6, label='Avg Price per Order (RSD)')
    
    ax2.set_ylabel('Number of Orders', color='darkred')
    ax2.tick_params(axis='y', labelcolor='darkred')
    ax3.set_ylabel('Average Price (RSD)', color='green')
    ax3.tick_params(axis='y', labelcolor='green')
    
    # Format average price y-axis
    ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    
    ax1.set_title('Monthly Wolt Analysis: Spending, Orders, and Average Price')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    lines3, labels3 = ax3.get_legend_handles_labels()
    ax1.legend(lines1 + lines2 + lines3, labels1 + labels2 + labels3, 
               loc='upper left')
    
    plt.tight_layout()
    plt.show()

# Run analysis
result = analyze_monthly_data(orders)
spending_dict, spending_list, count_dict, count_list, avg_dict, avg_list = result

# Choose one of the plotting options:
# Option 1: Two separate subplots (cleaner, easier to read)
plot_triple_analysis(spending_list, count_list, avg_list)
